# NYC Taxi — PySpark practice

استعملت المشروع هذا للتدريب من بيانات من كيقل. عددها ١٠ مليون،
وسويت جداول سلفر وقولد.
وبغيت امارس بشكل اكبر سبارك ف هذا من اسباب استعمالي له

 للملاحظة، البيانات ضخمة والشخص اللي يبي يحملها يحصلها ب كيقل والرابط موجود في README 
 النوت بوك جاهزة للاستعمال والاستعلام للي يبي يحللها 

Raw → Profiling / Data Quality → Silver → SQL Analytics → Gold → Zone Join

Put `taxi_trip_data.csv` and `taxi_zone_geo.csv` in `data/`. Run the cells in order. Paths work from the project folder or `notebooks/`. Generated Parquet folders go in `output/`.

The attached notebook contained Raw and Data Quality only. The later sections here were restored from the project conversation; see `REVIEW_NOTES.md`.


## 1. Raw data


In [ ]:
from pathlib import Path
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, when, abs, to_date, date_format, year

PROJECT_ROOT = Path("..") if Path.cwd().name == "notebooks" else Path(".")
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "output"
silver_path = str(OUTPUT_DIR / "silver" / "taxi_trips")
gold_dir = OUTPUT_DIR / "gold"

spark = SparkSession.builder.appName("NYCTaxiPipeline").getOrCreate()


In [ ]:
df = spark.read.csv(
    str(DATA_DIR / "taxi_trip_data.csv"),
    header=True,
    inferSchema=True  ## يتوقع وش نوع بيانات الكولمز
)
df.printSchema()


In [ ]:
df.select(
    "vendor_id",
    "pickup_datetime",
    "dropoff_datetime",
    "passenger_count",
    "trip_distance",
    "fare_amount"
).show(10, truncate=False)


## 2. Profiling / Data Quality

نشيك على البيانات أول قبل ما نقرر وش نحذف.


In [ ]:
total_rows = df.count()
print("Raw rows:", total_rows)
print("Columns:", len(df.columns))


In [ ]:
df.select([
    sum(
        when(col(c).isNull(), 1).otherwise(0)
    ).alias(c)
    for c in df.columns
]).show(vertical=True)


In [ ]:
unique_rows = df.dropDuplicates().count()
duplicate_rows = total_rows - unique_rows
print("Unique rows:", unique_rows)
print("Exact duplicate copies:", duplicate_rows)


In [ ]:
duplicate_groups = (
    df.groupBy(*df.columns)
    .count()
    .filter(col("count") > 1)
    .orderBy(col("count").desc() )
)


In [ ]:
duplicate_groups.show(3,truncate=False, vertical=True)


### Passengers and distance

القيم الغريبة تحتاج فحص؛ مو كل قيمة غريبة نحذفها.


In [ ]:
df.groupBy("passenger_count") \
    .count() \
    .orderBy("passenger_count") \
    .show()


In [ ]:
df.filter(
    col("passenger_count") == 0
).select(
    "pickup_datetime",
    "dropoff_datetime",
    "passenger_count",
    "trip_distance",
    "fare_amount",
    "total_amount"
).show(10, truncate=False)


In [ ]:
df.filter(
    col("passenger_count") > 6
).select(
    "pickup_datetime",
    "dropoff_datetime",
    "passenger_count",
    "trip_distance",
    "fare_amount",
    "total_amount"
).orderBy(col("passenger_count").desc()).show(10, truncate=False)


In [ ]:
df.select("trip_distance").summary(
    "min",
    "25%",
    "50%",
    "75%",
    "max"
).show()


In [ ]:
df.select(    
    "pickup_datetime",
    "dropoff_datetime",
    "passenger_count",
    "trip_distance",
    "fare_amount",
    "total_amount").orderBy(col("trip_distance").desc()).show(10)


In [ ]:
zero_distance_count = df.filter(col("trip_distance") == 0).count()
long_distance_count = df.filter(col("trip_distance") > 100).count()
print("Zero distance:", zero_distance_count)
print("Distance over 100:", long_distance_count)


### Trip duration

وقت النزول قبل الركوب غير منطقي. الرحلات بمدة صفر بنحتفظ فيها ونضيف flag.


In [ ]:
df_with_duration = df.withColumn(
    "trip_duration_minutes",
    (
        col("dropoff_datetime").cast("long")
        -
        col("pickup_datetime").cast("long")
    ) / 60
)


In [ ]:
df_with_duration.select(
    "pickup_datetime",
    "dropoff_datetime",
    "trip_distance",
    "fare_amount",
    "trip_duration_minutes"
).show(10, truncate=False)


In [ ]:
df_with_duration.select(
    "trip_duration_minutes"
).summary(
    "min",
    "25%",
    "50%",
    "75%",
    "max"
).show()


In [ ]:
invalid_duration_count = df_with_duration.filter(
    col("trip_duration_minutes") <= 0
).count()

print(
    "Trips with zero or negative duration:",
    invalid_duration_count
)


In [ ]:
df_with_duration.filter(
    col("trip_duration_minutes") <= 0
).select(
    "pickup_datetime",
    "dropoff_datetime",
    "trip_distance",
    "fare_amount",
    "trip_duration_minutes"
).orderBy(
    col("trip_duration_minutes")
).show(10, truncate=False)


In [ ]:
negative_duration_count = df_with_duration.filter(
    col("trip_duration_minutes") < 0
).count()

zero_duration_count = df_with_duration.filter(
    col("trip_duration_minutes") == 0
).count()

print("Negative duration:", negative_duration_count)
print("Zero duration:", zero_duration_count)


In [ ]:
duration_percentiles = df_with_duration.approxQuantile(
    "trip_duration_minutes",
    [0.90, 0.95, 0.99, 0.999],
    0.001
)

print("90%:", duration_percentiles[0])
print("95%:", duration_percentiles[1])
print("99%:", duration_percentiles[2])
print("99.9%:", duration_percentiles[3])


In [ ]:
longtrip_info = df_with_duration.filter(
    col("trip_duration_minutes") > 120
)
print("Trips longer than 120 minutes:", longtrip_info.count())


In [ ]:
longtrip_info.select(
    "pickup_datetime",
    "dropoff_datetime",
    "trip_duration_minutes",
    "trip_distance",
    "fare_amount",
    "total_amount"
).orderBy(
    col("trip_duration_minutes").desc()
).show(10, truncate=False)


In [ ]:
longAndZero = longtrip_info.filter(
    col("trip_distance") == 0
)

print("Zero distance and over 120 minutes:", longAndZero.count())


In [ ]:
longAndZero.select(
    "pickup_datetime",
    "dropoff_datetime",
    "trip_duration_minutes",
    "trip_distance",
    "fare_amount",
    "total_amount"
).orderBy(
    col("trip_duration_minutes").desc()
).show(10, truncate=False)


### Average speed

نحسب السرعة فقط إذا مدة الرحلة أكبر من صفر.


In [ ]:
df_with_speed = df_with_duration.withColumn(
    "average_speed",
    when(
        col("trip_duration_minutes") > 0,
        col("trip_distance") / (col("trip_duration_minutes") / 60)

    )
)


In [ ]:
df_with_speed.select(
    "pickup_datetime",
    "dropoff_datetime",
    "trip_duration_minutes",
    "trip_distance",
    "average_speed",
    "fare_amount"
).orderBy(
    col("average_speed").desc()
).show(10, truncate=False)


In [ ]:
high_speed_trips = df_with_speed.filter(
    col("average_speed") > 100
)

print(
    "Trips with average speed over 100:",
    high_speed_trips.count()
)


In [ ]:
high_speed_trips.select(
    "pickup_datetime",
    "dropoff_datetime",
    "trip_duration_minutes",
    "trip_distance",
    "average_speed",
    "fare_amount"
).orderBy(
    col("average_speed").desc()
).show(10, truncate=False)


### Fares

نفحص الأسعار السالبة والصفرية، بدون حذفها تلقائي.


In [ ]:
df.select("fare_amount").summary(
    "min",
    "25%",
    "50%",
    "75%",
    "max"
).show()


In [ ]:
negativeFare = df.filter(
col("fare_amount") < 0
).count()


zeroFare = df.filter(
col("fare_amount") == 0
).count()

print("Negative fare:", negativeFare)
print("Zero fare:", zeroFare)


In [ ]:
df.filter(
    col("fare_amount") < 0
).select(
    "pickup_datetime",
    "dropoff_datetime",
    "trip_distance",
    "fare_amount",
    "total_amount",
    "payment_type"
).orderBy(
    col("fare_amount")
).show(10, truncate=False)


## 3. Silver

نحذف النسخ المكررة والمدة السالبة.

 نخلي الرحلات بمدة صفر، والركاب صفر، والمسافة صفر مع flags.
 
  الأسعار والحالات المشبوهة تبقى للتحليل.

The original rule is `trip_duration_minutes >= 0`. It also excludes null durations. Flags use `1` for a matching condition and `0` otherwise; they do not replace the null checks above. يعني اذا طلع بالنتيجة واحد معناه الفلاق متفعل عليه



In [ ]:
silver_df = (
    df_with_speed
    .dropDuplicates()
    .filter(col("trip_duration_minutes") >= 0)
    .withColumn("zero_passenger_flag", when(col("passenger_count") == 0, 1).otherwise(0))
    .withColumn("zero_distance_flag", when(col("trip_distance") == 0, 1).otherwise(0))
    .withColumn("zero_duration_flag", when(col("trip_duration_minutes") == 0, 1).otherwise(0))
    .withColumn(
        "financial_mismatch_flag",
        when(
            abs(col("total_amount") - (
                col("fare_amount") + col("extra") + col("mta_tax")
                + col("tip_amount") + col("tolls_amount") + col("imp_surcharge")
            )) > 0.01,
            1
        ).otherwise(0)
    )
    .withColumn(
        "payment_type_name",
        when(col("payment_type") == 1, "Credit Card")
        .when(col("payment_type") == 2, "Cash")
        .when(col("payment_type") == 3, "No Charge")
        .when(col("payment_type") == 4, "Dispute")
        .when(col("payment_type") == 5, "Unknown")
        .when(col("payment_type") == 6, "Voided Trip")
        .otherwise("Other")
    )
    .withColumn("pickup_date", to_date(col("pickup_datetime")))
    .withColumn("pickup_month", date_format(col("pickup_datetime"), "yyyy-MM"))
)
silver_rows = silver_df.count()
print("Silver rows:", silver_rows)
print("Removed rows:", total_rows - silver_rows)


### ملاحظة: تواريخ خارج 2018

في التشغيل اللي قبل لقينا **162 رحلة بتاريخ ركوب خارج 2018**. 

قررنا نخليها في المشروع التدريبي.

 الفحص اللي تحت يحسب عددها بدون حذفها، لذلك ملخص الأشهر في Gold ممكن يشمل أشهر خارج 2018.



In [ ]:
invalid_year_trips = silver_df.filter(year(col("pickup_datetime")) != 2018)
print("Trips outside 2018:", invalid_year_trips.count())
invalid_year_trips.groupBy(
    year(col("pickup_datetime")).alias("pickup_year")
).count().orderBy("pickup_year").show(20, truncate=False)


### Save Silver as Parquet

Build Silver from the raw CSV, then write and read it back. This avoids the previous issue with reading a Parquet folder and overwriting that same folder from a DataFrame that still depends on it. ا


In [ ]:
silver_df.write.mode("overwrite").parquet(silver_path)
silver_check = spark.read.parquet(silver_path)
print("Saved Silver rows:", silver_check.count())


## 4. SQL Analytics

Each summary answers one question: trips and amounts by month, by payment type, and by pickup location. `total_amount` is the total trip amount, not a defined company revenue metric. These summaries include the retained suspicious records.


In [ ]:
silver_check.createOrReplaceTempView("taxi_trips")


In [ ]:
monthly_summary = spark.sql("""
SELECT
    pickup_month,
    COUNT(*) AS total_trips,
    ROUND(SUM(total_amount), 2) AS total_amount,
    ROUND(AVG(total_amount), 2) AS avg_trip_amount
FROM taxi_trips
GROUP BY pickup_month
ORDER BY pickup_month
""")
monthly_summary.show(30, truncate=False)


In [ ]:
payment_summary = spark.sql("""
SELECT
    payment_type_name,
    COUNT(*) AS total_trips,
    ROUND(SUM(total_amount), 2) AS total_amount,
    ROUND(AVG(total_amount), 2) AS avg_trip_amount
FROM taxi_trips
GROUP BY payment_type_name
ORDER BY total_trips DESC
""")
payment_summary.show(10, truncate=False)


In [ ]:
pickup_summary = spark.sql("""
SELECT
    pickup_location_id,
    COUNT(*) AS total_trips,
    ROUND(AVG(total_amount), 2) AS avg_trip_amount
FROM taxi_trips
GROUP BY pickup_location_id
ORDER BY total_trips DESC
""")
pickup_summary.show(10, truncate=False)


## 5. Gold

نحفظ الملخصات كجداول Parquet، كل جدول له مسار مستقل.


In [ ]:
monthly_summary.write.mode("overwrite").parquet(str(gold_dir / "monthly_summary"))
payment_summary.write.mode("overwrite").parquet(str(gold_dir / "payment_summary"))
pickup_summary.write.mode("overwrite").parquet(str(gold_dir / "pickup_location_summary"))


## 6. ربط مناطق اللي ناخذ العميل بأسمائها

نربط ملخص مناطق الالتقاط بملف المناطق عشان نضيف اسم المنطقة والـborough لكل ID.

نستخدم `left join` حتى نحتفظ بكل مناطق الالتقاط، حتى لو ما لقينا لها تطابق في ملف المناطق.

نعرض الأعمدة اللي نحتاجها فقط، ونترك `zone_geom` لأنه يحتوي نصًا طويلًا عن حدود المنطقة.

In [ ]:
zone_df = spark.read.csv(
    str(DATA_DIR / "taxi_zone_geo.csv"), header=True, inferSchema=True
).select("zone_id", "zone_name", "borough")
zone_df.show(10, truncate=False)

pickup_with_zone = (
    pickup_summary
    .join(zone_df, pickup_summary["pickup_location_id"] == zone_df["zone_id"], "left")
    .select("pickup_location_id", "zone_name", "borough", "total_trips", "avg_trip_amount")
)
# Re-sort after the join to display the top pickup zones.
pickup_with_zone.orderBy(
    col("total_trips").desc()
    ).show(10, truncate=False)
    
pickup_with_zone.write.mode("overwrite").parquet(
    str(gold_dir / "pickup_location_summary_enriched")
)
